# Step  1: Environment Variable Setup & Path Resolution

In [ ]:
import os
import sys
import runpy
from pathlib import Path

# --- 1. Determine Project Root ---
if (Path.cwd() / "pyproject.toml").exists():
    PROJ_ROOT = Path.cwd()
elif (Path.cwd().parent / "pyproject.toml").exists():
    PROJ_ROOT = Path.cwd().parent
else:
    # Fallback: assume notebook is in 'notebooks/'
    PROJ_ROOT = Path().resolve().parent 

# --- 2. Add config directory to sys.path ---
# Your ls output shows 'ai_weather', but you mentioned 'al_weather'. We check both.
CONFIG_DIR = PROJ_ROOT / "ai_weather"
if not CONFIG_DIR.exists():
    CONFIG_DIR = PROJ_ROOT / "al_weather"
    
if str(CONFIG_DIR) not in sys.path:
    sys.path.insert(0, str(CONFIG_DIR))

# --- 3. Import config and set ROMP paths ---
import config

ROMP_ROOT = config.ROMP_ROOT.resolve() if hasattr(config, 'ROMP_ROOT') else PROJ_ROOT
MOMP_PKG_DIR = config.MOMP_PKG_DIR.resolve() if hasattr(config, 'MOMP_PKG_DIR') else PROJ_ROOT / "momp"

# Add ROMP_ROOT and PROJ_ROOT to sys.path so local imports (like momp) work
for p in [str(ROMP_ROOT), str(PROJ_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import momp
import momp.params as params
from momp.lib.loader import get_cfg


2026-09-23 15:50:06.377 | INFO     | config:<module>:11 - PROJ_ROOT path is: /Users/mesfind/opt/Research/uchicago/ETAgroMetAIx/HCWF/ETAgroMetAIx/Training/ai-weather/ai_weather


package base dir /Users/mesfind/opt/Research/uchicago/ETAgroMetAIx/HCWF/ETAgroMetAIx/Training/ai-weather/ai_weather/ai_weather/benchmarking/ROMP/momp
work_dir =  /Users/mesfind/opt/Research/uchicago/ETAgroMetAIx/HCWF/ETAgroMetAIx/Training/ai-weather/ai_weather/notebooks/apps/benchmarking/ROMP
pkg_dir =  /Users/mesfind/opt/Research/uchicago/ETAgroMetAIx/HCWF/ETAgroMetAIx/Training/ai-weather/ai_weather/notebooks/apps/benchmarking/ROMPmomp
ref_model_dir =  /Users/mesfind/opt/Research/uchicago/ETAgroMetAIx/HCWF/ETAgroMetAIx/Training/ai-weather/ai_weather/notebooks/data/external/ENACTS
out_dir =  /Users/mesfind/opt/Research/uchicago/data/ROMP_OUT/et/output
out_fig =  /Users/mesfind/opt/Research/uchicago/data/ROMP_OUT/et/figure
obs_dir =  /Users/mesfind/opt/Research/uchicago/ETAgroMetAIx/HCWF/ETAgroMetAIx/Training/ai-weather/ai_weather/notebooks/data/external/ENACTS_regridded_025
thresh_file =  None
shpfile_dir =  /Users/mesfind/opt/Research/uchicago/ETAgroMetAIx/HCWF/ETAgroMetAIx/Training/a

In [10]:
def execute_momp_benchmark(mode="det"):
    print("=" * 60)
    print(f"Initializing MOMP Benchmark Execution [{mode.upper()} MODE]")
    print("=" * 60)
    
    config.setup_earthdata_auth()
    
    # --- ROBUST METADATA PATCH FOR LOCAL UNINSTALLED PACKAGE ---
    import importlib.metadata
    from importlib.metadata import PackageNotFoundError
    
    original_version = importlib.metadata.version
    original_distribution = importlib.metadata.distribution
    
    class DummyDistribution:
        version = "0.0.1"
        
    def patched_version(pkg_name):
        if pkg_name.lower() == "momp":
            return "0.0.1"
        try:
            return original_version(pkg_name)
        except PackageNotFoundError:
            return "0.0.1"
            
    def patched_distribution(pkg_name):
        if pkg_name.lower() == "momp":
            return DummyDistribution()
        return original_distribution(pkg_name)
        
    importlib.metadata.version = patched_version
    importlib.metadata.distribution = patched_distribution
    # -----------------------------------------------------------
    
    # Load configuration from config.in
    cfg = get_cfg()
    
    # --- CRITICAL FIX: Correct the missing slash in pkg_dir (e.g., 'ROMPmomp') ---
    # We force it to use the correctly resolved MOMP_PKG_DIR from Cell 1.
    cfg.pkg_dir = str(MOMP_PKG_DIR)
    
    # Automatically resolve relative paths from config.in against project root
    cfg.work_dir = resolve_path(getattr(cfg, "work_dir", str(ROMP_ROOT)))
    cfg.obs_dir = resolve_path(getattr(cfg, "obs_dir", "data/external/ENACTS_regridded_025"))
    cfg.ref_model_dir = resolve_path(getattr(cfg, "ref_model_dir", "data/external/ENACTS"))
    cfg.nc_mask = resolve_path(getattr(cfg, "nc_mask", "data/external/jjas_100mm_rainfall_mask_0p25.nc"))
    
    if hasattr(cfg, "shpfile_dir") and cfg.shpfile_dir:
        cfg.shpfile_dir = resolve_path(cfg.shpfile_dir)
        
    cfg.dir_out = str(config.ROMP_DEMO_OUT_DIR)
    cfg.dir_fig = str(config.ROMP_DEMO_FIG_DIR)
    
    # Configure mode-specific settings
    configure_benchmark_mode(cfg, mode=mode)
    
    # Ensure model directories are fully resolved
    cfg.model_dir_list = tuple(resolve_path(d) for d in cfg.model_dir_list)
    
    # Synchronize parameters into momp.params for worker processes
    for attr in dir(cfg):
        if not attr.startswith("_"):
            setattr(params, attr, getattr(cfg, attr))
            
    os.makedirs(cfg.dir_out, exist_ok=True)
    os.makedirs(cfg.dir_fig, exist_ok=True)
    
    print(f"work_dir      : {cfg.work_dir}")
    print(f"pkg_dir       : {cfg.pkg_dir}") # Verify this now ends in /momp
    print(f"obs_dir       : {cfg.obs_dir}")
    print(f"Probabilistic : {cfg.probabilistic}")
    print(f"Model List    : {cfg.model_list}")
    print(f"Model Paths   : {cfg.model_dir_list}")
    print("-" * 60)
    
    driver_path = MOMP_PKG_DIR / "driver.py"
    if driver_path.exists():
        print(f"Running MOMP driver pipeline at {driver_path}...")
        try:
            runpy.run_path(str(driver_path), run_name="__main__")
            print(f"\nMOMP [{mode.upper()}] benchmarking completed successfully.")
        finally:
            importlib.metadata.version = original_version
            importlib.metadata.distribution = original_distribution
    else:
        print(f"Error: Driver not found at {driver_path}")

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    mode_dropdown = widgets.Dropdown(
        options=['det',  'prob'],
        value='det',
        description='Mode:',
    )
    
    run_button = widgets.Button(description="Run MOMP Benchmark", button_style='primary')
    
    def on_button_clicked(b):
        print(f"\nSelected mode: {mode_dropdown.value}")
        execute_momp_benchmark(mode=mode_dropdown.value)
        
    run_button.on_click(on_button_clicked)
    
    display(widgets.HBox([mode_dropdown, run_button]))
    print("Select the evaluation mode and click 'Run MOMP Benchmark'.")
    
except ImportError:
    print("ipywidgets not found. Running with default MODE variable.")
    MODE = "det"  # Change this to "prob" for probabilistic mode
    execute_momp_benchmark(mode=MODE)

Select the evaluation mode and click 'Run MOMP Benchmark'.
